# Enterprise Clinical Risk Assessment System (10-Year Diabetes Profile)

### Objective
Building models for heavily imbalanced clinical fields requires strict safeguards against the 'Accuracy Paradox'. This notebook structures an advanced tree-boosting pipeline utilizing XGBoost to identify patient readmission flags, eliminating training data drops and pandas dummy state mismatches.

In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Ingest and clean database transaction logs chronological tracking parameters
dataset_url = "https://raw.githubusercontent.com/andrewwlong/diabetes_readmission/master/diabetic_data.csv"
raw_clinical_records = pd.read_csv(dataset_url)

# Deduplicate records by sorting chronologically and holding only a patient's first initial entry
raw_clinical_records.sort_values('encounter_id', inplace=True)
raw_clinical_records.drop_duplicates(subset=['patient_nbr'], keep='first', inplace=True)

# Map our target variable around the severe 30-day readmission window
features_scope = [
    'race', 'gender', 'age', 'time_in_hospital', 'medical_specialty',
    'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient',
    'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed'
]

X = raw_clinical_records[features_scope]
y = raw_clinical_records['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. Clinical Domain Knowledge Translation
Instead of using hundreds of raw high-cardinality alphanumeric ICD-9 codes, we write an explicit medical diagnostic grouping transformer to bucket related health issues together, stabilizing features for model consistency.

In [39]:
from sklearn.base import BaseEstimator, TransformerMixin

class ClinicalDiagnosisTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        processed_df = pd.DataFrame(X).copy()
        processed_df = processed_df.replace('?', np.nan)

        for classification_column in ['diag_1', 'diag_2', 'diag_3']:
            if classification_column in processed_df.columns:
                processed_df[classification_column] = processed_df[classification_column].apply(self._group_icd9_codes)
        return processed_df

    @staticmethod
    def _group_icd9_codes(code):
        if pd.isnull(code):
            return 'Unknown'
        clean_code = str(code).strip()
        if clean_code.startswith(('E', 'V')) or clean_code == '?':
            return 'Injury'
        try:
            numeric_prefix = float(clean_code.split('.')[0])
        except ValueError:
            return 'Other'

        if 390 <= numeric_prefix <= 459 or numeric_prefix == 785:
            return 'Circulatory'
        elif 460 <= numeric_prefix <= 519 or numeric_prefix == 786:
            return 'Respiratory'
        elif 520 <= numeric_prefix <= 579 or numeric_prefix == 787:
            return 'Digestive'
        elif numeric_prefix == 250:
            return 'Diabetes'
        elif 800 <= numeric_prefix <= 999:
            return 'Injury'
        elif 710 <= numeric_prefix <= 739:
            return 'Musculoskeletal'
        elif 580 <= numeric_prefix <= 629 or numeric_prefix == 788:
            return 'Genitourinary'
        elif 140 <= numeric_prefix <= 239:
            return 'Neoplasms'
        else:
            return 'Other'

## 2. Structural Column Imputation & Gradient Boosting
We isolate numerical and categorical streams inside an immutable `ColumnTransformer` layout to avoid row-dropping errors during production tracking, then optimize with an class-weighted XGBoost model.

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
import joblib
from google.colab import files

num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient']
cat_cols = ['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3',
            'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed']

# Structural preprocessing strategies
numerical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
                                 ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))])

processing_gate = ColumnTransformer([('num', numerical_pipeline, num_cols), ('cat', categorical_pipeline, cat_cols)])

# Calculate class distribution weights to adjust loss metrics natively
imbalance_multiplier = float(y_train.value_counts()[0] / y_train.value_counts()[1])

clinical_boosting_pipeline = Pipeline([
    ('icd9_mapping', ClinicalDiagnosisTransformer()),
    ('processing_matrices', processing_gate),
    ('regressor', XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05,
                                scale_pos_weight=imbalance_multiplier, random_state=42, n_jobs=-1))
])

print("Training high-performance clinical boosting framework...")
clinical_boosting_pipeline.fit(X_train, y_train)

# Evaluation Performance Tracking
y_pred = clinical_boosting_pipeline.predict(X_test)
y_prob = clinical_boosting_pipeline.predict_proba(X_test)[:, 1]

print("\n--- Model Evaluation Completed ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Performance Score: {roc_auc_score(y_test, y_prob):.4f}")

# Export Pipeline
joblib.dump(clinical_boosting_pipeline, 'diabetes_pipeline.pkl')
files.download('diabetes_pipeline.pkl')

Training high-performance clinical boosting framework...

--- Model Evaluation Completed ---
              precision    recall  f1-score   support

           0       0.94      0.62      0.75     13045
           1       0.12      0.56      0.20      1259

    accuracy                           0.62     14304
   macro avg       0.53      0.59      0.48     14304
weighted avg       0.86      0.62      0.70     14304

ROC-AUC Performance Score: 0.6231


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>